# Polymarket Data Check

Quick validation for the first Polymarket metadata and YES-token price-history pull.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
markets_path = ROOT / 'data' / 'processed' / 'markets.parquet'
prices_path = ROOT / 'data' / 'processed' / 'prices_long.parquet'
panel_path = ROOT / 'data' / 'processed' / 'price_panel.parquet'

In [ ]:
markets = pd.read_parquet(markets_path)
prices = pd.read_parquet(prices_path)
panel = pd.read_parquet(panel_path) if panel_path.exists() else pd.DataFrame()

summary = {
    'markets_collected': len(markets),
    'markets_with_usable_yes_history': prices['market_id'].nunique() if not prices.empty else 0,
    'price_rows': len(prices),
    'timestamp_min': prices['timestamp'].min() if not prices.empty else None,
    'timestamp_max': prices['timestamp'].max() if not prices.empty else None,
    'panel_shape': panel.shape,
    'panel_missing_fraction': float(panel.isna().mean().mean()) if not panel.empty else None,
}
summary

In [ ]:
markets['category'].fillna('Unknown').value_counts().head(25)

In [ ]:
if not prices.empty:
    prices.groupby('market_id').size().describe()
else:
    print('No price histories collected yet.')

In [ ]:
if not prices.empty:
    sample_ids = prices.groupby('market_id').size().sort_values(ascending=False).head(5).index
    fig, ax = plt.subplots(figsize=(12, 6))
    for market_id in sample_ids:
        sample = prices[prices['market_id'] == market_id].sort_values('timestamp')
        ax.plot(sample['timestamp'], sample['yes_price'], label=str(market_id))
    ax.set_title('Sample YES-token price histories')
    ax.set_ylabel('YES implied probability')
    ax.set_xlabel('Timestamp')
    ax.legend(title='market_id', loc='best')
    plt.tight_layout()
else:
    print('No price histories collected yet.')